In [3]:
import sys
sys.path.append("..")
from src.features.building_height import *
from src.features.nested_functions import *

import fiona
import pandas as pd
import geopandas as gpd
import os
from datetime import datetime


pd.set_option('display.max_columns', None)



Town names

In [ ]:
mmc_town_names = ['Arlington', 'Boston', 'Braintree', 'Brookline', 'Cambridge', 'Chelsea', 
             'Everett', 'Malden', 'Medford', 'Melrose', 'Newton', 'Quincy', 'Revere', 
             'Somerville', 'Watertown', 'Winthrop']


Run ArcPy Lidar processing / raster functions to create ndsm rasters for buildings in each MMC town (bounding box).

*only run again if necessary - takes a while and ndsm has been run for each town in MMC already*


In [6]:
las_folder = 'I:\Imagery\MassGIS_LAS_files'
town_name = 'Lynn'

in_las = r"I:\Imagery\MassGIS_LAS_files\19TCG337699.laz"
target_folder = r"I:\Imagery\MassGIS_LAS_files"
out_las_dataset = os.path.join(las_folder, (town_name + '_lasd.lasd'))
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference("NAD 1983 StatePlane Massachusetts FIPS 2001 (Meters)")

lynn_las_dataset = arcpy.conversion.ConvertLas(in_las=in_las, 
                            target_folder=target_folder,
                            out_las_dataset=out_las_dataset, 
                            )

ndsm_raster = create_ndsm_raster(town_name=town_name,
                                las_dataset=lynn_las_dataset)

dtm layer creation


ExecuteError: Failed to execute. Parameters are not valid.
ERROR 000732: Input LAS Dataset: Dataset I:\Imagery\MassGIS_LAS_files\Lynn_lasd.lasd does not exist or is not supported
WARNING 000725: Output Layer: Dataset Lynn_ground_layer already exists.
Failed to execute (MakeLasDatasetLayer).


In [4]:
#run again only if need be!



las_folder = 'I:\Imagery\MassGIS_LAS_files'

for town_name in ["Lynn"]:
    
    print(town_name + ' processing starting at ' + str(datetime.now()))


    #create las dataset 
    las_dataset = create_las_dataset(town_name = town_name, 
                                     las_folder=las_folder) 
    
    #create an ndsm raster
    ndsm_raster = create_ndsm_raster(town_name=town_name,
                                    las_dataset=las_dataset)
    


Lynn processing starting at 2025-06-24 15:58:00.803141
dtm layer creation
dsm layer creation
ndsm layer creation


RuntimeError: ERROR 000871: Output Raster: Unable to delete the output \\data-sync\public\DataServices\Projects\Current_Projects\Neighborhood_Planning_and_Zoning\Zoning_Projects\Rightsizing_Zoning_2025\Rightsizing_Zoning_ndsm.gdb\Lynn_ndsm_buildings.

Run building height function to do zonal statistics - for each town, generates a building footprint layer with summary statistics for raster cells within each footprint boundary

*only run again if necessary - takes a while and  has been run for each town in MMC already*

In [5]:
#only run again if necessary

for town_name in ["Lynn"]:
    
    print(town_name + ' processing starting at ' + str(datetime.now()))

    #RUN BUILDING HEIGHT FUNCTION
    stories = get_building_heights_from_ndsm(town_name = town_name)

Lynn processing starting at 2025-06-24 13:59:29.339735


ExecuteError: Failed to execute. Parameters are not valid.
ERROR 010568: Invalid extent. Please check for zero length or width, or failure to project extent to output spatial reference. 
Failed to execute (ZonalStatisticsAsTable).


Finally, put all of the footprint layers together and enrich with "stories" (meters / 3.3), land parcel info, and "flat_roof" field from Cool Roofs layer. Exports to a gdb feature layer and returns a geodataframe.

In [ ]:
# create mmc-wide gdf of 

cool_roofs_fp = r'K:\DataServices\Projects\Current_Projects\Climate_Change\MVP_MMC_CoolRoofs_MVP\Data\Analysis_Data\Data_Cool_Roofs\2_Output\MMC_Cool_Roofs.shp'
cool_roofs_gdf = gpd.read_file(cool_roofs_fp)

path = r"\\data-sync\public\DataServices\Projects\Current_Projects"
gdb_path = os.path.join(path, 'Neighborhood_Planning_and_Zoning\Zoning_Projects\Rightsizing_Zoning_2025\RightsizingZoning_2025.gdb')
output_layer_name = '00_mmc_enriched_structures'
list_of_town_names = mmc_town_names

mmc_footprints_with_height_and_flat = run_building_height_process(gdb_path=gdb_path, 
                                                                list_of_town_names=list_of_town_names,
                                                                output_layer_name=output_layer_name,
                                                                cool_roofs_gdf=cool_roofs_gdf)


c:\Users\RBowers\AppData\Local\ESRI\conda\envs\geospatial-env\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
c:\Users\RBowers\AppData\Local\ESRI\conda\envs\geospatial-env\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: Geometry of polygon of fid 12717 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(
